# Study of the multivariate distribution shift

## *Experience description*

## Experiment 2 - Multivariate Patch-Based Distribution Analysis

In the first experiment, the distribution shift between climate scenarios was analyzed independently for each variable. While this univariate approach provides a physically interpretable view of changes in mean, variability, extremes, and higher-order moments, it does not capture the joint structure of the variables or their interdependencies.

To address this limitation, the second experiment adopts a multivariate perspective. We study the distribution shift of sixteen CMIP6 variables from the CESM2 model (tas, ta850, ta500, huss, hus850, hus500, ua850, va850, ua500, va500, wap500, zg500, pr, psl, rsds, sfcWind) across five climates (historical, SSP126, SSP245, SSP370, SSP585), using a patch-based representation.

Specifically, we restrict the analysis to a configurable latitude band (default: ±30°) and split the data into ~1000 km × 1000 km spatial patches. Each sample corresponds to a multivariate patch at a given time, containing all selected variables over all grid points within the patch. Patches are represented as fixed-size grids and flattened in a consistent order, ensuring that each vector position corresponds to the same relative location within the patch. This provides a common coordinate system across all samples, enabling direct comparison while preserving local spatial structure.

This representation allows us to focus on the distribution of local multivariate climate patterns, rather than absolute geographic locations, and ensures compatibility with standard machine learning methods such as PCA and autoencoders.

To analyze the multivariate distribution, we project the data into a lower-dimensional space using Principal Component Analysis (PCA). The PCA is fitted on standardized data (mean 0, unit variance), computed globally across all climates, variable by variable, and grid point by grid point. This ensures a common reference space and allows meaningful comparison between climates. A sufficient number of components is retained to explain more than 90% of the total variance.

PCA provides an orthogonal basis capturing the dominant modes of variability, including correlations between variables, while reducing dimensionality and improving computational stability. It also enables a consistent framework for comparing both the original data and learned representations (e.g., latent spaces from autoencoders).

Distribution shifts between climates are then analyzed in PCA space using the same metrics as in the univariate case (moments, quantiles, and distribution distances), now applied to the principal components. This allows us to characterize changes in the joint structure of the data across climate scenarios.

Overall, this multivariate analysis complements the initial univariate study by capturing both individual variable behavior and the structure of their interactions, providing a more complete view of distribution shift.

### Part 0 - Global Configuration

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.path as mpath
from matplotlib.patches import PathPatch
from scipy import stats
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import to_rgba
from IPython.display import display
from cycler import cycler
import seaborn as sns
import json
from pathlib import Path

**Style**

In [ ]:
# Reset to a clean baseline, then apply a global scientific style.
plt.style.use("default")
sns.set_theme(style="whitegrid", context="notebook", font="DejaVu Sans")

_plot_alpha = 0.8
_base_palette = [to_rgba(color, alpha=_plot_alpha) for color in ["#2F3B52", "#226592", "#CC7B39", "#A33124", "#6B7280", "#4B5563"]]
# BAR_CORNER_RADIUS is in display pixels (visual radius).
# Use a small integer like 6 for a subtle, consistent rounding on all bars regardless of height.
BAR_CORNER_RADIUS = 150
plt.rcParams.update({
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "savefig.facecolor": "#FFFFFF",
    "savefig.transparent": False,
    "font.family": "DejaVu Sans",
    "font.sans-serif": ["DejaVu Sans", "Liberation Sans", "Arial"],
    "mathtext.fontset": "stix",
    "axes.edgecolor": "#B8BEC7",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#D9DDE3",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.65,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "semibold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.edgecolor": "#D9DDE3",
    "legend.fontsize": 10,
    "figure.figsize": (10.5, 4.8),
    "figure.dpi": 110,
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
    "axes.prop_cycle": cycler(color=_base_palette),
    "image.cmap": "Blues",
    "figure.constrained_layout.use": True,
})

def _round_bar_patches(ax, rounding_size=None):
    """Round the TOP corners of bar patches with a fixed visual radius.
    Parameters:
    - ax: matplotlib Axes containing bar patches.
    - rounding_size: visual radius in display pixels (int or float). If None, uses BAR_CORNER_RADIUS.

    This implementation converts the requested pixel radius to data coordinates locally for each bar
    so the visual rounding appears the same for all bars (controlled in pixels), independent of bar height.
    """
    rounding_px = BAR_CORNER_RADIUS if rounding_size is None else float(rounding_size)
    rounding_px = max(float(rounding_px), 0.0)
    kappa = 0.5522847498307936

    for patch in list(ax.patches):
        if not hasattr(patch, "get_x"):
            continue

        width = float(patch.get_width())
        height = float(patch.get_height())
        if width == 0 or height == 0:
            continue

        x0 = float(patch.get_x())
        y0 = float(patch.get_y())
        x1 = x0 + width
        y1 = y0 + height
        left = min(x0, x1)
        right = max(x0, x1)
        bottom = min(y0, y1)
        top = max(y0, y1)

        # Convert visual pixel radius to data coordinates at the lower-left corner of the bar
        try:
            disp_origin = ax.transData.transform((left, bottom))
            disp_x = disp_origin + np.array([rounding_px, 0])
            disp_y = disp_origin + np.array([0, rounding_px])
            data_x = ax.transData.inverted().transform(disp_x)
            data_y = ax.transData.inverted().transform(disp_y)
            data_origin = ax.transData.inverted().transform(disp_origin)
            data_radius_x = abs(data_x[0] - data_origin[0])
            data_radius_y = abs(data_y[1] - data_origin[1])
            radius_data = min(data_radius_x, data_radius_y)
        except Exception:
            # Fallback: treat rounding_px as data units if transform fails
            radius_data = rounding_px

        radius = min(radius_data, (right - left) / 2.0, (top - bottom) / 2.0)
        if radius <= 0:
            continue

        if height >= 0:
            verts = [
                (left, bottom),
                (right, bottom),
                (right, top - radius),
                (right, top - radius + kappa * radius),
                (right - radius + kappa * radius, top),
                (right - radius, top),
                (left + radius, top),
                (left + radius - kappa * radius, top),
                (left, top - radius + kappa * radius),
                (left, top - radius),
                (left, bottom),
            ]
        else:
            verts = [
                (left, top),
                (right, top),
                (right, bottom + radius),
                (right, bottom + radius - kappa * radius),
                (right - radius + kappa * radius, bottom),
                (right - radius, bottom),
                (left + radius, bottom),
                (left + radius - kappa * radius, bottom),
                (left, bottom + radius - kappa * radius),
                (left, bottom + radius),
                (left, top),
            ]

        codes = [
            mpath.Path.MOVETO,
            mpath.Path.LINETO,
            mpath.Path.LINETO,
            mpath.Path.CURVE4,
            mpath.Path.CURVE4,
            mpath.Path.CURVE4,
            mpath.Path.LINETO,
            mpath.Path.CURVE4,
            mpath.Path.CURVE4,
            mpath.Path.CURVE4,
            mpath.Path.CLOSEPOLY,
        ]

        rounded = PathPatch(
            mpath.Path(verts, codes),
            facecolor=patch.get_facecolor(),
            edgecolor=patch.get_edgecolor(),
            linewidth=patch.get_linewidth(),
            linestyle=patch.get_linestyle(),
            alpha=patch.get_alpha(),
            zorder=patch.get_zorder(),
            hatch=patch.get_hatch(),
            fill=patch.get_fill(),
            transform=ax.transData,
        )
        ax.add_patch(rounded)
        patch.set_visible(False)

**Data Loading**

You have to run a PBS jobs to train and run models and to extract latent representations of different climate test sets.

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
num_sample = 100000
variable = None  # Use None to analyze all variables

In [ ]:
precomputed_dir = Path(
    f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}"
)

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}

metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch

features_by_climate = {}
label_variable_by_climate = None
label_variable = None

if variable is None:
    # No target variable: keep all variables as model features.
    selected_variables = selected_variables_full.copy()

    for c in climate_order:
        X_full = features_by_climate_full[c]

        if X_full.shape[1] != expected_dim_full:
            raise ValueError(
                f"Unexpected feature dimension for {c}: "
                f"got {X_full.shape[1]}, expected {expected_dim_full}."
            )

        features_by_climate[c] = X_full.copy()

else:
    # Split off the specified variable as a dedicated label while keeping sample/grid-point alignment.
    if variable not in selected_variables_full:
        raise ValueError(
            f"Variable '{variable}' was not found in run_config selected_variables."
        )

    variable_var_index = selected_variables_full.index(variable)
    variable_col_start = variable_var_index * grid_points_per_patch
    variable_col_end = variable_col_start + grid_points_per_patch

    feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
    feature_mask_without_variable[variable_col_start:variable_col_end] = False

    label_variable_by_climate = {}

    for c in climate_order:
        X_full = features_by_climate_full[c]

        if X_full.shape[1] != expected_dim_full:
            raise ValueError(
                f"Unexpected feature dimension for {c}: "
                f"got {X_full.shape[1]}, expected {expected_dim_full}."
            )

        # Keep the specified variable values as label.
        label_variable_by_climate[c] = X_full[
            :, variable_col_start:variable_col_end
        ].copy()

        # Keep all non-target variables as model features.
        features_by_climate[c] = X_full[:, feature_mask_without_variable]

    selected_variables = [v for v in selected_variables_full if v != variable]

    # Convenience aggregate preserving same sample order as stacked climate features.
    label_variable = np.vstack(
        [label_variable_by_climate[c] for c in climate_order]
    )

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(
    precomputed_dir / "sampling_diagnostics.csv"
).set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")

sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)

if variable is None:
    print(f"Feature variables used downstream: {selected_variables}")
    print("No label variable created because variable is None.")
else:
    print(f"Feature variables used downstream without {variable}: {selected_variables}")
    print(f"label_{variable} shape, all samples x grid points: {label_variable.shape}")

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()

# Harmonized climate palette for all figures
preferred_climate_colors = {
    "historical": to_rgba("#2F3B52", alpha=_plot_alpha),  # dark navy-grey
    "ssp245": to_rgba("#2C7FB8", alpha=_plot_alpha),      # blue
    "ssp370": to_rgba("#F28E2B", alpha=_plot_alpha),      # orange
    "ssp585": to_rgba("#C0392B", alpha=_plot_alpha),      # deep red
}
for _climate_name, _color in preferred_climate_colors.items():
    if _climate_name in climate_order:
        climate_colors[_climate_name] = _color

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step builds one unified multivariate dataset used by all downstream analyses.

Workflow:
- select data inside +/- max_abs_lat (default: 30 deg);
- split the region into non-overlapping ~1000 km x 1000 km patches;
- for each climate, build samples from all variables and all lat/lon points inside each patch at sampled times;
- aggregate all climates to fit one common scaler

We measure the number of samples available and the grid size. And we obtain the following results :

- Patch grid shape (lat x lon): 10 x 7
- Grid points per patch: 70
- Potential sample counts per climate (before stride and max_samples_per_climate):

| scenario   | n_time_before_stride | n_patches | n_samples_before_stride_and_cap |
|------------|----------------------|-----------|---------------------------------|
| historical | 23725                | 246       | 5836350                         |
| ssp126     | 10950                | 246       | 2693700                         |
| ssp245     | 14600                | 246       | 3591600                         |
| ssp370     | 18250                | 246       | 4489500                         |
| ssp585     | 31390                | 246       | 7721940                         |

**PCA basis**

The normalization is performed globally across all climate scenarios by fitting a StandardScaler on the pooled dataset. This ensures that all climates are represented in a common feature space, allowing for consistent comparison of their distributions.

This approach preserves relative differences between climates while avoiding artificial alignment that would arise from climate-specific normalization. It is particularly important for multivariate analysis and PCA, where a shared reference frame is required to interpret distribution shifts.

In [ ]:
n_pca_components =10   # number of PCA components to retain for dimensionality reduction

In [ ]:
# Build one common PCA basis for all climates
X_all = np.vstack([features_by_climate[c] for c in climate_order])
meta_all = pd.concat([metadata_by_climate[c] for c in climate_order], ignore_index=True)

scaler = StandardScaler(with_mean=True, with_std=True)
X_all_scaled = scaler.fit_transform(X_all)

pca = PCA(n_components=n_pca_components, random_state=random_seed)
Z_all = pca.fit_transform(X_all_scaled)

# Normalize PCA space with global centroid and global RMS dispersion.
pca_global_centroid = np.mean(Z_all, axis=0)
Z_all_centered = Z_all - pca_global_centroid
pca_global_rms_dispersion = float(np.sqrt(np.mean(np.sum(Z_all_centered ** 2, axis=1))))
if pca_global_rms_dispersion <= 0:
    raise ValueError("Global PCA RMS dispersion is zero; normalization is undefined.")
Z_all = Z_all_centered / pca_global_rms_dispersion

# Split standardized features and PCA scores by climate
scaled_features_by_climate = {}
scores_by_climate = {}
start = 0
for climate in climate_order:
    n = features_by_climate[climate].shape[0]
    scaled_features_by_climate[climate] = X_all_scaled[start:start + n]
    scores_by_climate[climate] = Z_all[start:start + n]
    start += n

# Build a convenience DataFrame for plotting
score_frames = []
for climate in climate_order:
    Zc = scores_by_climate[climate]
    cols = {f"PC{i+1}": Zc[:, i] for i in range(Zc.shape[1])}
    frame = pd.DataFrame(cols)
    frame["scenario"] = climate
    frame["season"] = metadata_by_climate[climate]["season"].values
    score_frames.append(frame)

Explained Variance

In [ ]:
# Explained variance
scores_df = pd.concat(score_frames, ignore_index=True)
explained = pca.explained_variance_ratio_

sum = 0
print("PCA explained variance ratio:")
for i, ratio in enumerate(explained, start=1):
    print(f"PC{i}: {ratio:.3%}")
    sum += ratio

print(f"Total explained variance: {sum:.3%}")

fig, axes = plt.subplots(1, 1, figsize=(14, 5), constrained_layout=True)

# plot
axes.bar(np.arange(1, len(explained) + 1), explained * 100, color=climate_colors["historical"] )
# Temporary test: force a large rounding radius (pixels) to check visual effect
BAR_CORNER_RADIUS_TEST = 50
_round_bar_patches(axes, rounding_size=BAR_CORNER_RADIUS_TEST)
axes.plot(np.arange(1, len(explained) + 1), np.cumsum(explained) * 100, marker="o", color=climate_colors["ssp370"] )
axes.set_title("Explained variance by principal component")
axes.set_xlabel("Component")
axes.set_ylabel("Variance explained (%)")
axes.grid(alpha=0.25)

plt.show()

Explained variance by climate in the raw-data PCA projection

In [ ]:
# Explained variance of the raw-data PCA projection, computed separately for each climate.
# The PCA basis is shared; here we measure how much variance each climate carries along that basis.

climate_explained_variance = {}
climate_explained_cumulative = {}

for climate in climate_order:
    climate_scores = np.asarray(scores_by_climate[climate])
    component_variances = np.var(climate_scores, axis=0, ddof=0)
    total_variance = float(np.sum(component_variances))
    if total_variance <= 0:
        raise ValueError(f"Variance is zero for climate '{climate}' in the PCA projection.")
    climate_explained_variance[climate] = component_variances / total_variance
    climate_explained_cumulative[climate] = np.cumsum(climate_explained_variance[climate]) * 100

fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
component_index = np.arange(1, len(explained) + 1)

climate_linestyles = {
    "historical": "-",
    "ssp245": "--",
    "ssp370": "-.",
    "ssp585": ":",
}

for climate in climate_order:
    ax.plot(
        component_index,
        climate_explained_cumulative[climate],
        marker="o",
        markersize=4.5,
        linewidth=2.4,
        linestyle=climate_linestyles.get(climate, "-"),
        color=climate_colors[climate],
        alpha=0.95,
    )

legend_handles = [
    Line2D(
        [0], [0],
        color=climate_colors[climate],
        linestyle=climate_linestyles.get(climate, "-"),
        marker="o",
        markersize=5,
        linewidth=2.4,
        label=climate,
    )
    for climate in climate_order
]

ax.set_title("Cumulative explained variance by climate in the raw-data PCA projection")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance captured (%)")
ax.set_xlim(1, len(explained))
first_component_min = min(float(values[0]) for values in climate_explained_cumulative.values())
ax.set_ylim(max(0.0, first_component_min - 2.0), 100)
ax.axhline(90, color="black", linestyle="--", linewidth=1.0, alpha=0.7)
ax.grid(alpha=0.25)
ax.legend(handles=legend_handles, title="Climate", ncol=2, frameon=True)

plt.show()

We choose enough components in order to have more than 90% variance explained

### Part I — PCA Representation of Climate States

Since each sample is high-dimensional (all variables x all grid points in one patch), PCA provides a compact geometric view.

How to read this panel:
- overlap between clouds implies similar multivariate distributions;
- shifted centroids indicate a change in mean state;
- spread and orientation changes indicate covariance/shape changes.

PC1 / PC2

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5.5), constrained_layout=True)

# Left: PC1-PC2 with climate centroids
for climate in climate_order:
    data = scores_df[scores_df["scenario"] == climate]
    axes.scatter(data["PC1"], data["PC2"], s=10, alpha=0.55, color=climate_colors[climate], label=climate, zorder=1)
    cx = data["PC1"].mean()
    cy = data["PC2"].mean()
    axes.scatter([cx], [cy], s=180, marker="X", color=climate_colors[climate], edgecolor="black", linewidth=1.2, zorder=20)
axes.set_title("Multivariate climate samples in PCA space")
axes.set_xlabel("PC1")
axes.set_ylabel("PC2")
axes.grid(alpha=0.55)
axes.legend()

plt.show()

Because a great part of the variance is explained by other components of the PCA we plot some other settings :

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5.5), constrained_layout=True)

plot_data = scores_df[scores_df["scenario"].isin(climate_order)].copy()
plot_data = plot_data.sample(frac=1, random_state=42)  # mixing climates

for climate in climate_order:
    data = plot_data[plot_data["scenario"] == climate]
    ax.scatter(
        data["PC1"], data["PC2"],
        s=6,
        alpha=0.18,
        color=climate_colors[climate],
        label=climate,
        rasterized=True,
        zorder=1
    )

for climate in climate_order:
    data = scores_df[scores_df["scenario"] == climate]
    cx = data["PC1"].mean()
    cy = data["PC2"].mean()
    ax.scatter(
        [cx], [cy],
        s=220,
        marker="X",
        color=climate_colors[climate],
        edgecolor="black",
        linewidth=1.4,
        zorder=20
    )

ax.set_title("Multivariate climate samples in PCA space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

In [ ]:

from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

fig, ax = plt.subplots(1, 1, figsize=(15, 4), constrained_layout=True)

x_min = scores_df["PC1"].quantile(0.001)
x_max = scores_df["PC1"].quantile(0.999)
x_grid = np.linspace(x_min, x_max, 600)

ridge_height = 0.3
vertical_spacing = 0.1

for i, climate in enumerate(climate_order):
    data = scores_df.loc[scores_df["scenario"] == climate, "PC1"].dropna()

    kde = gaussian_kde(data)
    density = kde(x_grid)
    density = density / density.max() * ridge_height

    y_offset = i * vertical_spacing
    color = climate_colors[climate]

    ax.fill_between(
        x_grid,
        y_offset,
        y_offset + density,
        color=color,
        alpha=0.60,
        linewidth=0
    )

    ax.plot(
        x_grid,
        y_offset + density,
        color=color,
        linewidth=1.8
    )

    ax.hlines(
        y_offset,
        x_min,
        x_max,
        color="black",
        linewidth=0.6,
        alpha=0.20
    )

    # Mean
    mean_pc1 = data.mean()
    ax.vlines(
        mean_pc1,
        0,
        y_offset + ridge_height,
        color=color,
        linestyle="-",
        linewidth=1.5,
        alpha=0.95
    )

    # Extreme quantiles
    quantiles = {
        "q01": data.quantile(0.01),
        "q05": data.quantile(0.05),
        "q95": data.quantile(0.95),
        "q99": data.quantile(0.99),
    }

    for q_value in quantiles.values():
        ax.vlines(
            q_value,
            0,
            y_offset + ridge_height,
            color=color,
            linestyle="--",
            linewidth=1.2,
            alpha=0.85
        )

ax.set_yticks(np.arange(len(climate_order)) * vertical_spacing)
ax.set_yticklabels(climate_order)

ax.set_title("Distribution of PC1 across climate scenarios")
ax.set_xlabel("PC1")
ax.set_ylabel("Scenario")

ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right", "left"]].set_visible(False)

legend_elements = [
    Line2D([0], [0], color="black", linestyle="-", linewidth=1.5, label="Mean"),
    Line2D([0], [0], color="black", linestyle="--", linewidth=1.2, label="1st, 5th, 95th, 99th percentiles"),
]

ax.legend(handles=legend_elements, title="Reference lines", loc="upper right")

plt.show()

   

PC1 / PC3

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5.5), constrained_layout=True)

# Left: PC1-PC2 with climate centroids
for climate in climate_order:
    data = scores_df[scores_df["scenario"] == climate]
    axes.scatter(data["PC1"], data["PC3"], s=10, alpha=0.55, color=climate_colors[climate], label=climate, zorder=1)
    cx = data["PC1"].mean()
    cy = data["PC3"].mean()
    axes.scatter([cx], [cy], s=180, marker="X", color=climate_colors[climate], edgecolor="black", linewidth=1.2, zorder=20)
axes.set_title("Multivariate climate samples in PCA space")
axes.set_xlabel("PC1")
axes.set_ylabel("PC3")
axes.grid(alpha=0.55)
axes.legend()

plt.show()

PC2 / PC3

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5.5), constrained_layout=True)

# Left: PC1-PC2 with climate centroids
for climate in climate_order:
    data = scores_df[scores_df["scenario"] == climate]
    axes.scatter(data["PC2"], data["PC3"], s=10, alpha=0.55, color=climate_colors[climate], label=climate, zorder=1)
    cx = data["PC2"].mean()
    cy = data["PC3"].mean()
    axes.scatter([cx], [cy], s=180, marker="X", color=climate_colors[climate], edgecolor="black", linewidth=1.2, zorder=20)
axes.set_title("Multivariate climate samples in PCA space")
axes.set_xlabel("PC2")
axes.set_ylabel("PC3")
axes.grid(alpha=0.55)
axes.legend()

plt.show()

PC4 / PC5

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5.5), constrained_layout=True)

# Left: PC1-PC2 with climate centroids
for climate in climate_order:
    data = scores_df[scores_df["scenario"] == climate]
    axes.scatter(data["PC4"], data["PC5"], s=10, alpha=0.55, color=climate_colors[climate], label=climate, zorder=1)
    cx = data["PC4"].mean()
    cy = data["PC5"].mean()
    axes.scatter([cx], [cy], s=180, marker="X", color=climate_colors[climate], edgecolor="black", linewidth=1.2, zorder=20)
axes.set_title("Multivariate climate samples in PCA space")
axes.set_xlabel("PC4")
axes.set_ylabel("PC5")
axes.grid(alpha=0.55)
axes.legend()

plt.show()

To study to what extent the PCA space is configured by the seasonal cycle, we represent a seasonal coloring on the historical climate :

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5.5), constrained_layout=True)

# Right: seasonal coloring on historical as an example of within-climate structure
season_palette = {"DJF": "#2F3B52", "MAM": "#5B738B", "JJA": "#F28E2B", "SON": "#C0392B"}
hist = scores_df[scores_df["scenario"] == "historical"]
for season, color in season_palette.items():
    season_data = hist[hist["season"] == season]
    axes.scatter(season_data["PC1"], season_data["PC2"], s=10, alpha=0.45, color=color, label=season, zorder=1)
    cx = season_data["PC1"].mean()
    cy = season_data["PC2"].mean()
    
    axes.scatter(
        [cx], [cy],
        s=185, marker="X",
        color=color,
        edgecolor="black",
        linewidth=1.2,
        zorder=20
    )
axes.set_title("Historical seasonal structure in PCA space")
axes.set_xlabel("PC1")
axes.set_ylabel("PC2")
axes.grid(alpha=0.55)
axes.legend()

plt.show()

### Part II — Multivariate Shift Metrics in PCA Space

These metrics are computed in PCA space, using the scores of the standardized patch features, and they are computed between all possible pairs of climate :
- centroid distance between climate means in PCA space;
- KS distance computed component by component, then aggregated with mean and max; 
- Wasserstein distance computed component by component, then aggregated with mean and max;
- sliced Wasserstein distance obtained from random 1D projections in PCA space.

We compute Centroid, KS and Wasserstein distances :

In [ ]:
ssp_scenarios = [c for c in climate_order if c != "historical"]

def per_component_distances(x_ref: np.ndarray, x_tgt: np.ndarray):
    """Compute KS and Wasserstein distances for each PCA component."""
    ks_vals = []
    wass_vals = []
    n_components = x_ref.shape[1]
    for k in range(n_components):
        ks_stat = ks_2samp(x_ref[:, k], x_tgt[:, k]).statistic
        wass = wasserstein_distance(x_ref[:, k], x_tgt[:, k])
        ks_vals.append(float(ks_stat))
        wass_vals.append(float(wass))
    return np.array(ks_vals), np.array(wass_vals)

# Pairwise metrics in PCA space (all climate pairs)
pairwise_rows = []
component_rows = []

for climate_i in climate_order:
    Xi = scores_by_climate[climate_i]
    mu_i = np.mean(Xi, axis=0)
    for climate_j in climate_order:
        Xj = scores_by_climate[climate_j]
        mu_j = np.mean(Xj, axis=0)

        if climate_i == climate_j:
            ks_vals = np.zeros(Xi.shape[1], dtype=float)
            wass_vals = np.zeros(Xi.shape[1], dtype=float)
            centroid_dist = 0.0
        else:
            ks_vals, wass_vals = per_component_distances(Xi, Xj)
            centroid_dist = float(np.linalg.norm(mu_i - mu_j))

        pairwise_rows.append({
            "climate_i": climate_i,
            "climate_j": climate_j,
            "n_i": int(Xi.shape[0]),
            "n_j": int(Xj.shape[0]),
            "centroid_distance": centroid_dist,
            "ks_mean": float(np.mean(ks_vals)),
            "ks_max": float(np.max(ks_vals)),
            "wasserstein_mean": float(np.mean(wass_vals)),
            "wasserstein_max": float(np.max(wass_vals)),
        })

        if climate_i == "historical":
            for comp_idx, (ks_v, w_v) in enumerate(zip(ks_vals, wass_vals), start=1):
                component_rows.append({
                    "scenario": climate_j,
                    "component": f"PC{comp_idx}",
                    "ks": float(ks_v),
                    "wasserstein": float(w_v),
                })

pairwise_pca_metrics_df = pd.DataFrame(pairwise_rows)
component_metrics_df = pd.DataFrame(component_rows)

# Summary table relative to historical
comparison_pca_df = (
    pairwise_pca_metrics_df[pairwise_pca_metrics_df["climate_i"] == "historical"]
    .set_index("climate_j")
    .loc[climate_order, ["n_j", "centroid_distance", "ks_mean", "ks_max", "wasserstein_mean", "wasserstein_max"]]
    .rename(columns={"n_j": "n_samples"})
)

# Keep backward-compatible variable name used later in the notebook
comparison_df = comparison_pca_df.copy()

pca_distance_matrices = {
    metric: pairwise_pca_metrics_df.pivot(index="climate_i", columns="climate_j", values=metric).loc[climate_order, climate_order]
    for metric in ["centroid_distance", "ks_mean", "wasserstein_mean"]
}

Here is a small display of some of the values computed :

In [ ]:
print("PCA-space summary metrics relative to historical")
display(comparison_pca_df.style.format("{:.6g}"))

print("Per-component distances relative to historical")
display(component_metrics_df[component_metrics_df["scenario"].isin(ssp_scenarios)].head(12).style.format({col: "{:.6g}" for col in ["ks", "wasserstein"]}))


We now compute the sliced Wasserstein distance :

In [ ]:
def sliced_wasserstein_distance(x: np.ndarray, y: np.ndarray, n_projections: int = 128, seed: int = 42) -> float:
    """Approximate Wasserstein distance using random 1D projections."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    d = x.shape[1]
    projections = rng.normal(size=(n_projections, d))
    projections /= np.linalg.norm(projections, axis=1, keepdims=True) + 1e-12

    values = []
    for p in projections:
        xp = x @ p
        yp = y @ p
        values.append(float(wasserstein_distance(xp, yp)))

    return float(np.mean(values))

# Pairwise sliced Wasserstein matrix in PCA space
swd_rows = []
for climate_i in climate_order:
    Xi = scores_by_climate[climate_i]
    for climate_j in climate_order:
        Xj = scores_by_climate[climate_j]
        swd = 0.0 if climate_i == climate_j else sliced_wasserstein_distance(
            Xi, Xj, n_projections=128, seed=random_seed
        )
        swd_rows.append({
            "climate_i": climate_i,
            "climate_j": climate_j,
            "sliced_wasserstein": float(swd),
        })

pairwise_swd_df = pd.DataFrame(swd_rows)
swd_matrix = pairwise_swd_df.pivot(index="climate_i", columns="climate_j", values="sliced_wasserstein").loc[climate_order, climate_order]

swd_vs_historical_df = (
    pairwise_swd_df[pairwise_swd_df["climate_i"] == "historical"]
    .set_index("climate_j")
    .loc[climate_order, ["sliced_wasserstein"]]
)

Here is a display of some of the values computed :

In [ ]:
print("Sliced Wasserstein distance in PCA space (relative to historical)")
display(swd_vs_historical_df.style.format("{:.6g}"))

We now represent the four metrics for all possible pairs of climates using heatmaps :

In [ ]:
# Visualization of PCA-space distances between climates
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

heatmap_specs = [
    ("centroid_distance", "Centroid distance (PCA means)", axes[0, 0]),
    ("ks_mean", "Mean KS distance across PCs", axes[0, 1]),
    ("wasserstein_mean", "Mean Wasserstein distance across PCs", axes[1, 0]),
    ("sliced_wasserstein", "Sliced Wasserstein distance", axes[1, 1]),
]

for metric, title, ax in heatmap_specs:
    matrix = swd_matrix if metric == "sliced_wasserstein" else pca_distance_matrices[metric]
    im = ax.imshow(matrix.values, aspect="auto", cmap="Blues", vmax=matrix.values.max() * 0.85)
    ax.set_title(title)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.grid(False)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix.values[i, j]:.3f}", ha="center", va="center", fontsize=11, color="black", fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

In [ ]:
# Visualization of PCA-space distances between climates
fig, ax = plt.subplots(1, 1, figsize=(10, 8), constrained_layout=True)

metric = "wasserstein_mean"
title = "Mean Wasserstein distance across PCs"

matrix = swd_matrix if metric == "sliced_wasserstein" else pca_distance_matrices[metric]

im = ax.imshow(
    matrix.values,
    aspect="auto",
    cmap="Blues",
    vmax=matrix.values.max() * 0.85
)

ax.set_title(title)

ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels(matrix.columns, rotation=45, ha="right")

ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)

ax.grid(False)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(
            j,
            i,
            f"{matrix.values[i, j]:.3f}",
            ha="center",
            va="center",
            fontsize=11,
            color="black",
            fontweight="bold"
        )

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

We also focus on the metrics for the shift between SSPs and the historical climate :

In [ ]:
# historical vs SSP scenarios
plot_df = comparison_pca_df.loc[ssp_scenarios, ["centroid_distance", "ks_mean", "wasserstein_mean"]].copy()
plot_df["sliced_wasserstein"] = swd_vs_historical_df.loc[ssp_scenarios, "sliced_wasserstein"]

fig, ax = plt.subplots(1, 1, figsize=(10, 4), constrained_layout=True)

metric_colors = [climate_colors["historical"], climate_colors["ssp245"], climate_colors["ssp370"], climate_colors["ssp585"]]

plot_df.plot(kind="bar", ax=ax, color=metric_colors)
_round_bar_patches(ax, rounding_size=0.12)

ax.set_title("PCA-space distances vs historical")
ax.set_xlabel("Scenario")
ax.set_ylabel("Distance")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Metric")
plt.xticks(rotation=45, ha="right")

plt.show()

### Part III — Moments of Principal Components

To characterize climate projection shape changes in the reduced multivariate space, we inspect the moments of the first 3 principal components for each climate.

Interpretation:
- PC mean: directional shift along a principal component;
- PC std: spread or variability change;
- PC skew/kurtosis: asymmetry and tail-shape changes.

Here we compute the moments :

In [ ]:
pc_labels = [f"PC{i+1}" for i in range(min(3, n_pca_components))]

moment_rows = []
for climate in climate_order:
    X = scores_by_climate[climate]
    row = {"scenario": climate}
    for i, pc in enumerate(pc_labels):
        vals = X[:, i]
        row[f"{pc}_mean"] = float(np.mean(vals))
        row[f"{pc}_std"] = float(np.std(vals, ddof=0))
        row[f"{pc}_skew"] = float(stats.skew(vals, bias=False, nan_policy="omit"))
        row[f"{pc}_kurt"] = float(stats.kurtosis(vals, fisher=True, bias=False, nan_policy="omit"))
    moment_rows.append(row)

moments_df = pd.DataFrame(moment_rows).set_index("scenario").loc[climate_order]
moments_delta_df = moments_df.subtract(moments_df.loc["historical"], axis=1)

Here is a display of what we computed

In [ ]:
print("PC moments")
display(moments_df.style.format("{:.6g}"))

print("Differences vs historical")
display(moments_delta_df.style.format("{:+.6g}"))

We can represent the absolute values computed for the first component for example :

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, metric in zip(axes, [f"{pc_labels[0]}_mean", f"{pc_labels[0]}_std", f"{pc_labels[0]}_kurt"]):
    moments_df[metric].plot(kind="bar", ax=ax, color=[climate_colors[c] for c in climate_order])
    _round_bar_patches(ax, rounding_size=0.12)
    ax.set_title(metric)
    ax.grid(axis="y", alpha=0.25)
plt.show()

But it is more interesting to represent the shift between the historical climate and the SSPs. To aggregate all of this we use heatmaps, one for the mean and one for the standard deviation :

In [ ]:
# Heatmaps of moment shifts (vs historical) for the first three principal components
pc_triplet = ["PC1", "PC2", "PC3"]

mean_cols = [f"{pc}_mean" for pc in pc_triplet]
std_cols = [f"{pc}_std" for pc in pc_triplet]

mean_delta_matrix = moments_delta_df.loc[climate_order, mean_cols].copy()
mean_delta_matrix.columns = pc_triplet

std_delta_matrix = moments_delta_df.loc[climate_order, std_cols].copy()
std_delta_matrix.columns = pc_triplet

# Heatmap 1: mean deltas
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
mean_abs_max = np.nanmax(np.abs(mean_delta_matrix.values))
im = ax.imshow(mean_delta_matrix.values, aspect="auto", cmap="RdBu_r", vmin=-mean_abs_max, vmax=mean_abs_max)
ax.set_title("Delta Mean vs Historical in PCA Space")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Scenario")
ax.set_xticks(range(len(mean_delta_matrix.columns)))
ax.set_xticklabels(mean_delta_matrix.columns)
ax.set_yticks(range(len(mean_delta_matrix.index)))
ax.set_yticklabels(mean_delta_matrix.index)
ax.grid(False)
for i in range(mean_delta_matrix.shape[0]):
    for j in range(mean_delta_matrix.shape[1]):
        value = mean_delta_matrix.values[i, j]
        ax.text(j, i, f"{value:+.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Delta value")
plt.show()

# Heatmap 2: standard deviation deltas
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
std_abs_max = np.nanmax(np.abs(std_delta_matrix.values))
im = ax.imshow(std_delta_matrix.values, aspect="auto", cmap="RdBu_r", vmin=-std_abs_max, vmax=std_abs_max)
ax.set_title("Delta Standard Deviation vs Historical in PCA Space")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Scenario")
ax.set_xticks(range(len(std_delta_matrix.columns)))
ax.set_xticklabels(std_delta_matrix.columns)
ax.set_yticks(range(len(std_delta_matrix.index)))
ax.set_yticklabels(std_delta_matrix.index)
ax.grid(False)
for i in range(std_delta_matrix.shape[0]):
    for j in range(std_delta_matrix.shape[1]):
        value = std_delta_matrix.values[i, j]
        ax.text(j, i, f"{value:+.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Delta value")
plt.show()

### Part IV — Extreme Multivariate Scores

A simple multivariate extreme indicator is the PCA score norm.

For each sample, we compute ||z|| where z is the PCA score vector, so the extreme score is defined in PCA space.
Large values correspond to uncommon joint states in the reduced multivariate representation.

The next cell also decomposes extreme shifts component by component by looking at absolute PC values and their seasonal quantiles.

Here we compute the PCA score norm :

In [ ]:
extreme_rows = []
for climate in climate_order:
    Z = scores_by_climate[climate]
    norm = np.linalg.norm(Z, axis=1)
    extreme_rows.append(
        {
            "scenario": climate,
            "q95_norm": float(np.quantile(norm, 0.95)),
            "q99_norm": float(np.quantile(norm, 0.99)),
            "max_norm": float(np.max(norm)),
        }
    )

extremes_df = pd.DataFrame(extreme_rows).set_index("scenario").loc[climate_order]
extremes_delta_df = extremes_df.subtract(extremes_df.loc["historical"], axis=1)

Here is a display of what we computed :

In [ ]:
print("Extreme multivariate score levels")
display(extremes_df.style.format("{:.6g}"))

print("Differences vs historical")
display(extremes_delta_df.style.format("{:+.6g}"))

We represent the extremes absolute values and the shift from the historical climate

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
extremes_df[["q95_norm", "q99_norm"]].plot(kind="bar", ax=axes[0], color=[climate_colors[c] for c in climate_order])
_round_bar_patches(axes[0], rounding_size=0.12)
axes[0].set_title("95th and 99th percentiles of ||PCA score||")
axes[0].grid(axis="y", alpha=0.25)

extremes_delta_df[["q95_norm", "q99_norm"]].plot(kind="bar", ax=axes[1], color=[climate_colors[c] for c in climate_order])
_round_bar_patches(axes[1], rounding_size=0.12)
axes[1].set_title("Extreme-score shift vs historical")
axes[1].grid(axis="y", alpha=0.25)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 4), constrained_layout=True)

extremes_delta_df[["q95_norm", "q99_norm"]].plot(kind="bar", ax=axes, color=[climate_colors[c] for c in climate_order])
_round_bar_patches(axes, rounding_size=0.12)
axes.set_title("Extreme-score shift vs historical")
axes.grid(axis="y", alpha=0.25)

plt.show()

And we also represent the shift from the historical climate but for every component separately :

In [ ]:
# Per-component extreme-score shift vs historical in PCA space
n_components_for_extremes = n_pca_components
pc_names = [f"PC{i+1}" for i in range(n_components_for_extremes)]

component_extreme_rows = []
for climate in climate_order:
    Z = scores_by_climate[climate]
    for i, pc_name in enumerate(pc_names):
        vals = np.abs(Z[:, i])
        component_extreme_rows.append(
            {
                "scenario": climate,
                "component": pc_name,
                "q95_abs": float(np.quantile(vals, 0.95)),
                "q99_abs": float(np.quantile(vals, 0.99)),
            }
        )

component_extremes_df = pd.DataFrame(component_extreme_rows)

hist_component_extremes = (
    component_extremes_df[component_extremes_df["scenario"] == "historical"]
    .set_index("component")[["q95_abs", "q99_abs"]]
    .rename(columns={"q95_abs": "q95_abs_hist", "q99_abs": "q99_abs_hist"})
)

component_extreme_shift_df = component_extremes_df.join(
    hist_component_extremes, on="component", how="left"
    )
component_extreme_shift_df["q95_shift_vs_historical"] = (
    component_extreme_shift_df["q95_abs"] - component_extreme_shift_df["q95_abs_hist"]
)
component_extreme_shift_df["q99_shift_vs_historical"] = (
    component_extreme_shift_df["q99_abs"] - component_extreme_shift_df["q99_abs_hist"]
)

q95_shift_matrix = component_extreme_shift_df.pivot(
    index="scenario", columns="component", values="q95_shift_vs_historical"
).loc[climate_order, pc_names]
q99_shift_matrix = component_extreme_shift_df.pivot(
    index="scenario", columns="component", values="q99_shift_vs_historical"
).loc[climate_order, pc_names]

fig, axes = plt.subplots(2, 1, figsize=(max(10, n_components_for_extremes * 1.1), 7), constrained_layout=True)

q95_abs_max = np.nanmax(np.abs(q95_shift_matrix.values))
q99_abs_max = np.nanmax(np.abs(q99_shift_matrix.values))

im0 = axes[0].imshow(
    q95_shift_matrix.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-q95_abs_max,
    vmax=q95_abs_max,
)
axes[0].set_title("Per-component Q95 extreme-score shift vs historical")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Scenario")
axes[0].set_xticks(range(len(pc_names)))
axes[0].set_xticklabels(pc_names, rotation=45, ha="right")
axes[0].set_yticks(range(len(climate_order)))
axes[0].set_yticklabels(climate_order)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="Shift")

im1 = axes[1].imshow(
    q99_shift_matrix.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-q99_abs_max,
    vmax=q99_abs_max,
)
axes[1].set_title("Per-component Q99 extreme-score shift vs historical")
axes[1].set_xlabel("Principal Component")
axes[1].set_ylabel("Scenario")
axes[1].set_xticks(range(len(pc_names)))
axes[1].set_xticklabels(pc_names, rotation=45, ha="right")
axes[1].set_yticks(range(len(climate_order)))
axes[1].set_yticklabels(climate_order)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="Shift")

plt.show()

### Part V — Seasonal Multivariate Shift in PCA Space

Seasonal diagnostics below are computed in PCA space from the seasonal PCA scores.

For each scenario and season, we compare:
- the delta mean of PC1 relative to historical;
- the delta of the global 99th percentile of the PCA score norm relative to historical;
- the sliced Wasserstein distance relative to historical.

This provides a seasonal version of the multivariate PCA-based comparison used in the previous sections.

In [ ]:
season_order = ["DJF", "MAM", "JJA", "SON"]

seasonal_rows = []
for season in season_order:
    season_scores = {}
    for climate in climate_order:
        meta = metadata_by_climate[climate].reset_index(drop=True)
        Z = scores_by_climate[climate]
        mask = (meta["season"] == season).values
        season_scores[climate] = Z[mask]

    Z_hist = season_scores["historical"]
    if Z_hist.shape[0] == 0:
        for climate in climate_order:
            seasonal_rows.append({
                "season": season,
                "scenario": climate,
                "n": 0,
                "pc1_mean_delta_vs_historical": np.nan,
                "q99_norm_delta_vs_historical": np.nan,
                "sliced_wasserstein_vs_historical": np.nan,
            })
        continue

    hist_pc1_mean = float(np.mean(Z_hist[:, 0]))
    hist_q99_norm = float(np.quantile(np.linalg.norm(Z_hist, axis=1), 0.99))

    for climate in climate_order:
        Zc = season_scores[climate]
        if Zc.shape[0] == 0:
            seasonal_rows.append({
                "season": season,
                "scenario": climate,
                "n": 0,
                "pc1_mean_delta_vs_historical": np.nan,
                "q99_norm_delta_vs_historical": np.nan,
                "sliced_wasserstein_vs_historical": np.nan,
            })
            continue

        pc1_mean = float(np.mean(Zc[:, 0]))
        q99_norm = float(np.quantile(np.linalg.norm(Zc, axis=1), 0.99))

        if climate == "historical":
            swd = 0.0
        else:
            swd = sliced_wasserstein_distance(Z_hist, Zc, n_projections=128, seed=random_seed)

        seasonal_rows.append({
            "season": season,
            "scenario": climate,
            "n": int(Zc.shape[0]),
            "pc1_mean_delta_vs_historical": pc1_mean - hist_pc1_mean,
            "q99_norm_delta_vs_historical": q99_norm - hist_q99_norm,
            "sliced_wasserstein_vs_historical": float(swd),
        })

seasonal_df = pd.DataFrame(seasonal_rows)

pc1_delta_matrix = seasonal_df.pivot(index="season", columns="scenario", values="pc1_mean_delta_vs_historical").loc[season_order, climate_order]
q99_delta_matrix = seasonal_df.pivot(index="season", columns="scenario", values="q99_norm_delta_vs_historical").loc[season_order, climate_order]
swd_matrix_seasonal = seasonal_df.pivot(index="season", columns="scenario", values="sliced_wasserstein_vs_historical").loc[season_order, climate_order]



fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)

heatmap_specs = [
    (
        pc1_delta_matrix,
        "Delta mean vs historical for PC1 by season (PCA space)",
        "RdBu_r",
        True,
    ),
    (
        q99_delta_matrix,
        "Global 99th percentile shift vs historical by season (PCA space)",
        "RdBu_r",
        True,
    ),
    (
        swd_matrix_seasonal,
        "Sliced Wasserstein distance vs historical by season (PCA space)",
        "Blues",
        False,
    ),
]

for ax, (matrix, title, cmap, center_zero) in zip(axes, heatmap_specs):
    values = matrix.values
    if center_zero:
        vmax = np.nanmax(np.abs(values))
        vmin = -vmax
    else:
        vmin = np.nanmin(values)
        vmax = np.nanmax(values)

    im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.grid(False)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix.values[i, j]
            text = "NaN" if not np.isfinite(val) else (f"{val:+.2f}" if center_zero else f"{val:.2f}")
            ax.text(j, i, text, ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

### Part VI — Analyzing distribution shift

To complete